# Notebook 02 - Fintech Session Persistence Save/Restore

This notebook preserves and reviews Fintech session state after Notebook 00/01 runtime setup has created a real local workspace and session manifest. It is not a standalone first notebook.

Run Notebook 00 and Notebook 01 setup/session cells first so the runtime contains:

- `/content/fintech-market-ingestion-demo`
- `/content/fintech-market-ingestion-demo/artifacts/sessions/<REAL_SESSION_ID>/session_manifest.json`

Before any Drive folder creation, save dry-run, or live persistence action, replace `REPLACE_WITH_DRIVE_FOLDER_NAME` with an intentional Drive folder name and confirm `SESSION_ID` is not `REPLACE_WITH_SESSION_ID_IF_NEEDED`.

Generated session payloads, restore outputs, local app workspaces, generated data, archive packs, runtime folders, notebook outputs, and secrets must stay outside Git. Clear all outputs and keep execution counts `null` before committing notebook source.

Live save/restore and Drive operations remain manual Colab-only. Restore command behavior remains candidate/upstream-confirmation-dependent unless the upstream CLI and flags are confirmed.

## 1. Optional Colab Runtime Package Setup

Run this only in Google Colab when the upstream Fintech package is not already available. Local repository validation must not install packages, require network access, or mutate the runtime.

In [ ]:
# Manual Colab-only setup cell.
# Uncomment only when the Colab runtime needs the upstream package.
# !python -m pip install --upgrade pip
# !python -m pip install "pandas-market-calendars>=5.0"
# !python -m pip install -i https://test.pypi.org/simple/ fintech-market-ingestion

print("Package setup is manual Colab-only. Verify commands before live session save/restore.")

## 2. Verify Native Command Availability

These checks report whether expected upstream command entry points are visible in the current runtime. `fintech-save-session` is the known session persistence command from earlier notebooks. A dedicated restore/session command name must be confirmed against the current upstream CLI before M4.3 contract wiring.

In [ ]:
import shutil

COMMAND_CANDIDATES = [
    "fintech-init-project",
    "fintech-save-session",
    "fintech-restore-session",  # Candidate only; confirm upstream support before contract wiring.
]

for command_name in COMMAND_CANDIDATES:
    command_path = shutil.which(command_name)
    status = "available" if command_path else "not found"
    print(f"{command_name}: {status}")

## 3. Safe Help and Preview Cells

Help commands are safe candidates for later CLI contract validation when the upstream command exists locally. Restore command help is commented because the command name is not yet confirmed for M4.3.

In [ ]:
!fintech-save-session --help

# Candidate restore help only. Confirm the upstream command name before enabling.
# !fintech-restore-session --help

## 4. Define Portable Runtime Paths

Active app work stays under `/content`. Google Drive is used only as persistence and restore storage. Use shell-safe placeholders; avoid angle-bracket placeholders inside shell-interpolated paths.

In [ ]:
from pathlib import Path

CONTENT_ROOT = Path("/content")
FINTECH_ROOT = CONTENT_ROOT / "fintech-market-ingestion-demo"
SESSION_NAME = "colab_demo"

DRIVE_FOLDER_NAME = "REPLACE_WITH_DRIVE_FOLDER_NAME"
DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER_NAME
DRIVE_PROJECT_ROOT = DRIVE_ROOT / "fintech-market-ingestion"

SESSION_ID_PLACEHOLDER = "REPLACE_WITH_SESSION_ID_IF_NEEDED"

if not str(FINTECH_ROOT).startswith("/content/"):
    raise ValueError(f"Active Fintech workspace must stay under /content: {FINTECH_ROOT}")

if DRIVE_FOLDER_NAME == "REPLACE_WITH_DRIVE_FOLDER_NAME":
    print("Set DRIVE_FOLDER_NAME before mounting Drive or running persistence commands.")

print("FINTECH_ROOT:", FINTECH_ROOT)
print("DRIVE_PROJECT_ROOT:", DRIVE_PROJECT_ROOT)

## 5. Optional Google Drive Mount

Run this only in Google Colab when Drive persistence or restore storage is needed. Local validation must not mount Google Drive.

In [ ]:
# Manual Colab-only Drive mount cell.
# from google.colab import drive
# drive.mount("/content/drive")

print("Drive mount is manual Colab-only and is not required for local validation.")

## 6. Preserve or Recover `SESSION_ID`

Notebook 02 should use the session created by Notebook 00/01 runtime setup. This cell reads local session metadata when present and falls back to a shell-safe placeholder. Do not commit a real captured session ID from a prior runtime.

In [ ]:
import json

session_manifest_candidates = sorted(
    (FINTECH_ROOT / "artifacts" / "sessions").glob("*/session_manifest.json"),
    key=lambda path: path.stat().st_mtime if path.exists() else 0,
)

SESSION_MANIFEST = None
SESSION_ID = SESSION_ID_PLACEHOLDER

if session_manifest_candidates:
    session_manifest_path = session_manifest_candidates[-1]
    SESSION_MANIFEST = json.loads(session_manifest_path.read_text(encoding="utf-8"))
    SESSION_ID = str(SESSION_MANIFEST.get("session_id") or SESSION_ID_PLACEHOLDER)
    print("Loaded session metadata from runtime manifest.")
else:
    print("No local session manifest found. Use the SESSION_ID created by Notebook 00/01 setup.")

print("SESSION_ID:", SESSION_ID)

## 7. Prepare Drive Session Paths

Set `DRIVE_FOLDER_NAME` to a real, intentional Drive folder name before mounting Drive, creating folders, or previewing a session save. Do not use placeholder values for live Drive operations; placeholder-named Drive folders are not valid smoke-test evidence.

In [ ]:
DRIVE_SESSION_ROOT = DRIVE_PROJECT_ROOT / "sessions" / SESSION_ID
DRIVE_SESSION_RESTORE_ROOT = DRIVE_PROJECT_ROOT / "restore" / SESSION_ID

print("Drive session save root:", DRIVE_SESSION_ROOT)
print("Drive restore review root:", DRIVE_SESSION_RESTORE_ROOT)

if SESSION_ID == SESSION_ID_PLACEHOLDER:
    print("Replace SESSION_ID only at runtime, or run Notebook 00/01 setup to create one.")

## 8. Session Persistence Preflight

Run this non-mutating preflight before Drive folder creation, save dry-runs, or live save/restore actions. It blocks placeholder Drive/session values and verifies that Notebook 00/01 already prepared the expected Fintech workspace and runtime session manifest.

In [ ]:
SESSION_PERSISTENCE_PREFLIGHT_READY = True
preflight_messages = []

required_workspace_paths = [
    FINTECH_ROOT,
    FINTECH_ROOT / "configs",
    FINTECH_ROOT / "reports",
    FINTECH_ROOT / "artifacts",
    FINTECH_ROOT / "data" / "curated",
]

if DRIVE_FOLDER_NAME == "REPLACE_WITH_DRIVE_FOLDER_NAME":
    SESSION_PERSISTENCE_PREFLIGHT_READY = False
    preflight_messages.append("Set DRIVE_FOLDER_NAME before Drive persistence actions.")

if not FINTECH_ROOT.exists():
    SESSION_PERSISTENCE_PREFLIGHT_READY = False
    preflight_messages.append(f"Missing Fintech workspace root: {FINTECH_ROOT}")

for required_path in required_workspace_paths[1:]:
    if not required_path.exists():
        SESSION_PERSISTENCE_PREFLIGHT_READY = False
        preflight_messages.append(f"Missing expected workspace path: {required_path}")

if SESSION_ID == SESSION_ID_PLACEHOLDER:
    SESSION_PERSISTENCE_PREFLIGHT_READY = False
    preflight_messages.append("Run Notebook 00/01 setup to create a real SESSION_ID before saving.")

EXPECTED_SESSION_MANIFEST = FINTECH_ROOT / "artifacts" / "sessions" / SESSION_ID / "session_manifest.json"
if not EXPECTED_SESSION_MANIFEST.exists():
    SESSION_PERSISTENCE_PREFLIGHT_READY = False
    preflight_messages.append(f"Missing session manifest: {EXPECTED_SESSION_MANIFEST}")

for message in preflight_messages:
    print("BLOCKED:", message)

if SESSION_PERSISTENCE_PREFLIGHT_READY:
    print("Session persistence preflight ready.")
else:
    print("Session persistence preflight blocked. Fix blocked items before Drive folder creation or save dry-run.")

## 9. Manual Drive Folder Setup

Create Drive folders only in Colab after the preflight reports ready. This mutates mounted Drive storage and must remain excluded from local validation. Do not create folders while `DRIVE_FOLDER_NAME` or `SESSION_ID` still use placeholders.

In [ ]:
# Manual Colab-only persistence folder setup.
# Uncomment only after SESSION_PERSISTENCE_PREFLIGHT_READY is True.
# if not SESSION_PERSISTENCE_PREFLIGHT_READY:
#     raise RuntimeError("Session persistence preflight is not ready. Fix blocked items before creating Drive folders.")
# if DRIVE_FOLDER_NAME == "REPLACE_WITH_DRIVE_FOLDER_NAME":
#     raise RuntimeError("Set DRIVE_FOLDER_NAME before creating Drive folders.")
# if SESSION_ID == SESSION_ID_PLACEHOLDER:
#     raise RuntimeError("Run Notebook 00/01 setup to create a real SESSION_ID before creating Drive folders.")
# DRIVE_SESSION_ROOT.mkdir(parents=True, exist_ok=True)
# DRIVE_SESSION_RESTORE_ROOT.mkdir(parents=True, exist_ok=True)

print("Drive folder creation is manual Colab-only and blocked until session persistence preflight is ready.")

## 10. Review Local Workspace Readiness

This lightweight review checks expected local paths without printing broad generated data listings. It is safe for human review and should not be treated as evidence that generated artifacts are committed.

In [ ]:
EXPECTED_LOCAL_PATHS = [
    FINTECH_ROOT,
    FINTECH_ROOT / "configs",
    FINTECH_ROOT / "reports",
    FINTECH_ROOT / "artifacts",
    FINTECH_ROOT / "data" / "curated",
]

for path in EXPECTED_LOCAL_PATHS:
    print(f"{path}: exists={path.exists()}")

## 11. Preview Session Save

Start with a dry-run preview. The command should use the active local workspace and write only to the Drive session root when a live save is intentionally enabled. Confirm upstream dry-run behavior before adding this cell to M4.3 contract validation.

In [ ]:
import shlex
import subprocess

SAVE_DRY_RUN_COMMAND = [
    "fintech-save-session",
    "--root", str(FINTECH_ROOT),
    "--session-id", SESSION_ID,
    "--policy", "artifacts_and_reports",
    "--adapter", "google-drive",
    "--destination", str(DRIVE_SESSION_ROOT),
    "--dry-run",
]

print("Save dry-run command preview:")
print(" ".join(shlex.quote(part) for part in SAVE_DRY_RUN_COMMAND))

if not SESSION_PERSISTENCE_PREFLIGHT_READY:
    raise RuntimeError("Session persistence preflight is not ready. Do not run save dry-run yet.")

subprocess.run(SAVE_DRY_RUN_COMMAND, check=True)

## 12. Manual Session Save

Run this only after the dry run is reviewed and the Drive destination is intentional. This command can create session payloads in mounted Drive storage, so it must remain manual Colab-only and excluded from local validation.

In [ ]:
# Manual Colab-only live session save. Uncomment only after reviewing the dry run.
# !fintech-save-session #   --root "{FINTECH_ROOT}" #   --session-id "{SESSION_ID}" #   --policy artifacts_and_reports #   --adapter google-drive #   --destination "{DRIVE_SESSION_ROOT}"

## 13. Optional Curated Data Save Preview

Curated data can be large and is not the main Notebook 02 workflow. Keep this as an explicit opt-in dry-run preview only. Larger archive-oriented persistence belongs in Notebook 03.

In [ ]:
# Optional dry-run preview only; confirm upstream support before M4.3 contract wiring.
# !fintech-save-session #   --root "{FINTECH_ROOT}" #   --session-id "{SESSION_ID}" #   --policy curated_daily_bars #   --adapter google-drive #   --destination "{DRIVE_SESSION_ROOT}" #   --include-curated-data #   --dry-run

## 14. Restore Command Candidate Preview

A dedicated restore/session command name is not guaranteed by this repository. Confirm the current upstream `fintech-market-ingestion` CLI before enabling restore help, dry-run, or live restore cells in M4.3/M4.4.

In [ ]:
RESTORE_COMMAND_CANDIDATE = "fintech-restore-session"

restore_preview = " ".join([
    RESTORE_COMMAND_CANDIDATE,
    "--root", f'"{FINTECH_ROOT}"',
    "--adapter", "google-drive",
    "--source", f'"{DRIVE_SESSION_ROOT}"',
    "--dry-run",
])

print("Candidate restore dry-run command; confirm upstream CLI before running:")
print(restore_preview)

## 15. Manual Restore From Drive

Restore can overwrite or recreate runtime files under `/content`. Keep live restore commented until the upstream command name and flags are confirmed, Drive storage is mounted, and the target workspace has been reviewed.

In [ ]:
# Manual Colab-only restore candidate. Confirm upstream CLI before enabling.
# !fintech-restore-session #   --root "{FINTECH_ROOT}" #   --adapter google-drive #   --source "{DRIVE_SESSION_ROOT}"

# Optional force mode should be used only when replacement is intentional.
# !fintech-restore-session #   --root "{FINTECH_ROOT}" #   --adapter google-drive #   --source "{DRIVE_SESSION_ROOT}" #   --force

## 16. Lightweight Saved/Restored Session Review

After a manual save or restore, use small metadata checks rather than broad generated file listings. Do not paste copied manifests, generated payload listings, logs, screenshots, or restore outputs into committed notebook source.

In [ ]:
review_paths = {
    "local_workspace": FINTECH_ROOT,
    "local_reports": FINTECH_ROOT / "reports",
    "local_artifacts": FINTECH_ROOT / "artifacts",
    "drive_session_root": DRIVE_SESSION_ROOT,
}

for label, path in review_paths.items():
    print(f"{label}: {path} exists={path.exists()}")

if SESSION_MANIFEST:
    safe_manifest_keys = ["session_id", "session_name", "created_at"]
    safe_summary = {key: SESSION_MANIFEST.get(key) for key in safe_manifest_keys if key in SESSION_MANIFEST}
    print("Session manifest summary keys:", sorted(safe_summary))
else:
    print("No runtime session manifest loaded for review.")

## 17. Explicit Deferrals to Notebook 03+

Notebook 02 does not perform full archive backup or full archive restore. Defer these workflows to Notebook 03 and later notebooks:

- Full archive backup pack workflow.
- Full archive restore workflow.
- Archive shard/package inspection.
- Archive transfer workflow.
- Restore-pack execution workflow.
- StratLake initialization.
- Feature generation.
- Strategy smoke tests.
- Backtest review.

## 18. Cleanup Before Commit

Before committing Notebook 02 source:

- Clear all outputs.
- Keep every code cell `execution_count` as `null`.
- Confirm no real `SESSION_ID` value is committed.
- Confirm no private local paths, personal Drive folders, usernames, account names, credentials, tokens, or `.env` values are present.
- Confirm no generated session payloads, restore outputs, archive packs, generated data, local workspaces, runtime folders, logs, screenshots, or copied manifests are present.
- Keep Google Drive as persistence and restore storage only.
- Keep active app work under `/content`.
- Do not wire M4.3/M4.4 validation configs until the later issues.

## Notebook Summary

This cleaned Notebook 02 source preserves the session persistence workflow while keeping repository boundaries intact. It uses shell-safe placeholders, preserves `SESSION_ID` continuity through runtime metadata or a placeholder fallback, previews native session save behavior, marks live save/restore as manual Colab-only, and hands full archive workflows to Notebook 03+.